## Create DB file and load cloud extension

In [1]:
import duckdb, os, time

DB = os.path.expanduser("~/demo_taxi.duckdb")
con = duckdb.connect(DB)

con.sql("INSTALL httpfs; LOAD httpfs;")
print("duckdb", duckdb.__version__, "→", DB)


duckdb 2.0.0 → /home/jovyan/demo_taxi.duckdb


## Get NYC TAXI trips and zones dataset 

In [2]:
TRIPS_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

t= time.time()
con.sql(f"CREATE OR REPLACE TABLE trips AS SELECT * FROM read_parquet('{TRIPS_URL}')")
con.sql(f"CREATE OR REPLACE TABLE zones AS SELECT * FROM read_csv('{ZONES_URL}', header=true)")

print(f"trips: {con.sql('SELECT count(*) FROM trips').limit(1).fetchall()} rows  ({time.time()-t:.0f}s)")
print(f"zones: {con.sql('SELECT count(*) FROM zones').limit(1).fetchall()} rows")

con.sql("SELECT tpep_pickup_datetime, passenger_count, trip_distance, PULocationID, total_amount FROM trips LIMIT 10").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trips: [(2964624,)] rows  (3s)
zones: [(265,)] rows
┌──────────────────────┬─────────────────┬───────────────┬──────────────┬──────────────┐
│ tpep_pickup_datetime │ passenger_count │ trip_distance │ PULocationID │ total_amount │
│      timestamp       │      int64      │    double     │    int32     │    double    │
├──────────────────────┼─────────────────┼───────────────┼──────────────┼──────────────┤
│ 2024-01-01 00:57:55  │               1 │          1.72 │          186 │         22.7 │
│ 2024-01-01 00:03:00  │               1 │           1.8 │          140 │        18.75 │
│ 2024-01-01 00:17:06  │               1 │           4.7 │          236 │         31.3 │
│ 2024-01-01 00:36:38  │               1 │           1.4 │           79 │         17.0 │
│ 2024-01-01 00:46:51  │               1 │           0.8 │          211 │         16.1 │
│ 2024-01-01 00:54:08  │               1 │           4.7 │          148 │         41.5 │
│ 2024-01-01 00:49:44  │               2 │         10.82 │

## Demo queries and questions on NYC Taxi dataset

In [3]:
questions = {
    "Busiest pickup hour of the day?":
        """SELECT hour(tpep_pickup_datetime) AS hour, count(*) AS trips
           FROM trips GROUP BY 1 ORDER BY trips DESC LIMIT 5""",

    "Average tip percentage by passenger count (credit-card trips only)?":
        """SELECT passenger_count, count(*) AS trips,
                  round(100 * avg(tip_amount / nullif(fare_amount, 0)), 1) AS avg_tip_pct
           FROM trips WHERE payment_type = 1 AND fare_amount > 0 AND passenger_count BETWEEN 1 AND 6
           GROUP BY 1 ORDER BY 1""",

    "Top 5 pickup neighbourhoods by number of trips?":
        """SELECT z.Borough, z.Zone, count(*) AS trips
           FROM trips t JOIN zones z ON z.LocationID = t.PULocationID
           GROUP BY 1, 2 ORDER BY trips DESC LIMIT 5""",

    "How does the trip count change with a minimum-fare threshold?":
        """SELECT th.threshold, count(*) FILTER (WHERE total_amount >= th.threshold) AS trips
           FROM trips, (VALUES (10), (25), (50), (100), (200)) th(threshold)
           GROUP BY 1 ORDER BY 1""",

    "Trips from JFK to Manhattan: how many and what did they cost on average?":
        """SELECT count(*) AS trips, round(avg(total_amount), 2) AS avg_total, round(avg(trip_distance), 1) AS avg_miles
           FROM trips t JOIN zones pu ON pu.LocationID = t.PULocationID
                        JOIN zones d  ON d.LocationID  = t.DOLocationID
           WHERE pu.Zone = 'JFK Airport' AND d.Borough = 'Manhattan'""",
}

for q, sql in questions.items():
    print("\n##", q)
    con.sql(sql).show()


## Busiest pickup hour of the day?
┌───────┬────────┐
│ hour  │ trips  │
│ int64 │ int64  │
├───────┼────────┤
│    18 │ 212788 │
│    17 │ 206257 │
│    16 │ 190201 │
│    15 │ 189359 │
│    19 │ 184032 │
└───────┴────────┘


## Average tip percentage by passenger count (credit-card trips only)?
┌─────────────────┬─────────┬─────────────┐
│ passenger_count │  trips  │ avg_tip_pct │
│      int64      │  int64  │   double    │
├─────────────────┼─────────┼─────────────┤
│               1 │ 1812440 │        28.7 │
│               2 │  325701 │        25.7 │
│               3 │   71888 │        26.7 │
│               4 │   37587 │        24.8 │
│               5 │   27650 │        25.9 │
│               6 │   18502 │        25.9 │
└─────────────────┴─────────┴─────────────┘


## Top 5 pickup neighbourhoods by number of trips?


┌───────────┬───────────────────────┬────────┐
│  Borough  │         Zone          │ trips  │
│  varchar  │        varchar        │ int64  │
├───────────┼───────────────────────┼────────┤
│ Queens    │ JFK Airport           │ 145240 │
│ Manhattan │ Midtown Center        │ 143471 │
│ Manhattan │ Upper East Side South │ 142708 │
│ Manhattan │ Upper East Side North │ 136465 │
│ Manhattan │ Midtown East          │ 106717 │
└───────────┴───────────────────────┴────────┘


## How does the trip count change with a minimum-fare threshold?


┌───────────┬─────────┐
│ threshold │  trips  │
│   int32   │  int64  │
├───────────┼─────────┤
│        10 │ 2866163 │
│        25 │  965445 │
│        50 │  317079 │
│       100 │   40207 │
│       200 │    2013 │
└───────────┴─────────┘


## Trips from JFK to Manhattan: how many and what did they cost on average?
┌───────┬───────────┬───────────┐
│ trips │ avg_total │ avg_miles │
│ int64 │  double   │  double   │
├───────┼───────────┼───────────┤
│ 72167 │     92.58 │      18.8 │
└───────┴───────────┴───────────┘



## Check AWS S3 import to DuckDB 

In [4]:
con.sql(""" 
    CREATE OR REPLACE SECRET aws_anon (
        TYPE s3, PROVIDER config, KEY_ID '', SECRET '', REGION 'us-east-1'
    )
""")

S3_FILE = "s3://noaa-ghcn-pds/parquet/by_year/YEAR=2024/ELEMENT=TMAX/*.parquet"

t = time.time()
con.sql(f"CREATE OR REPLACE TABLE s3_check AS SELECT * FROM read_parquet('{S3_FILE}') LIMIT 1000")
print(f"s3_check: {con.sql('SELECT count(*) FROM s3_check').limit(1).fetchall} rows ({time.time()-t:.0f}s)")
con.sql("SELECT * FROM s3_check LIMIT 10").show()

s3_check: <bound method DuckDBPyRelation.fetchall of ┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1000 │
└──────────────┘
> rows (1s)
┌─────────────┬──────────┬────────────┬─────────┬─────────┬─────────┬──────────┬─────────┬───────┐
│     ID      │   DATE   │ DATA_VALUE │ M_FLAG  │ Q_FLAG  │ S_FLAG  │ OBS_TIME │ ELEMENT │ YEAR  │
│   varchar   │ varchar  │   int64    │ varchar │ varchar │ varchar │ varchar  │ varchar │ int64 │
├─────────────┼──────────┼────────────┼─────────┼─────────┼─────────┼──────────┼─────────┼───────┤
│ ASN00074148 │ 20240101 │        348 │ NULL    │ NULL    │ a       │ NULL     │ TMAX    │  2024 │
│ ASN00077094 │ 20240101 │        341 │ NULL    │ NULL    │ a       │ NULL     │ TMAX    │  2024 │
│ ASN00078015 │ 20240101 │        307 │ NULL    │ NULL    │ a       │ NULL     │ TMAX    │  2024 │
│ ASN00079097 │ 20240101 │        293 │ NULL    │ NULL    │ a       │ NULL     │ TMAX    │  2024 │
│ ASN00079099 │ 20240101 │        288 │ N

## Get Data from GCS public Buckets 

In [7]:
con.sql("CREATE OR REPLACE SECRET gcs_anon (TYPE gcs, KEY_ID '', SECRET '')")
con.sql("SET disable_parquet_prefetching = true")   # Google's sample file has quirky page offsets; this flag is only needed for it

GCS_FILE  = "gs://cloud-samples-data/bigquery/us-states/us-states.parquet"
GCS_HTTPS = "https://storage.googleapis.com/cloud-samples-data/bigquery/us-states/us-states.parquet"


try:
    con.sql(f"CREATE OR REPLACE TABLE gcs_check AS SELECT * FROM read_parquet('{GCS_FILE}')")
    print("gcs_check: read via gs:// {secret path works}")
except Exception as e: 
    print("gs:// failed:", str(e).splitlines()[0])
    con.sql(f"CREATE OR REPLACE TABLE gcs_check AS SELECT + FROM read_parquet('{GCS_HTTPS}')")
    print("gcs_check: read via HTTPS fallback worked")

print(f"gcs_check: {con.sql('SELECT count(*) FROM gcs_check').limit(1).fetchall()} rows")
con.sql("SELECT * FROM gcs_check LIMIT 100").show()

SyntaxError: invalid non-printable character U+00A0 (793751206.py, line 11)

## Done - close the connection and DuckDB file 

In [6]:
con.sql("SELECT table_name, estimated_size AS approx_rows FROM duckdb_tables() ORDER BY 1").show()
con.close()
if os.path.exists(DB):
    print("closed:", DB, f"({os.path.getsize(DB)/1e6:.0f} MB)")
else:
    print("WARNING: no file at", DB, "- the connection was not to that path; restart the kernel and Run All from section 0")


┌────────────┬─────────────┐
│ table_name │ approx_rows │
│  varchar   │    int64    │
├────────────┼─────────────┤
│ gcs_check  │          50 │
│ s3_check   │        1000 │
│ trips      │     2964624 │
│ zones      │         265 │
└────────────┴─────────────┘

closed: /home/jovyan/demo_taxi.duckdb (76 MB)


## TODO : Test writing to GCS Bucket as we have to write CxT data into parquet

In [10]:
# Need to set up bucket and credentials
GCS_TARGET_BUCKET = ""      # e.g. "bids-cxt-data"
GCS_KEY = os.environ.get("GCS_HMAC_KEY_ID", "")
GCS_SECRET - os.environ.get("GCS_HMAC_SECRET", "")

if GCS_TARGET_BUCKET and GCS_KEY:
    con.sql(f"CREATE OR REPLACE SECRET gcs_rw (TYPE gcs, KEY_ID '{GCS_KEY}', SECRET '{GCS_SECRET}')")
    con.sql(f"COPY zones TO 'gs://{GCS_TARGET_BUCKET}/demo/zones.parquet'( FORMAT parquet)")
    n = con.sql(f"SELECT count(*) FROM read_parquet('gs://{GCS_TARGET_BUCKET}/demo/zones.parquet')").fetchall()[0][0]
    print(f"wrote and read back gs://{MY_BUCKET}/demo/zones.parquet: {n} rows")
else:
    print("skipped (set MY_BUCKET and GCS_HMAC_KEY_ID / GCS_HMAC_SECRET to test writing to your bucket)")